In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append(".")
print(" Ready")

 Ready


In [2]:
%%writefile agents/summarization_agent.py
from collections import defaultdict
from typing import Dict, List

from tools.llm_clients import call_llm
from tools.logger import log_agent_event

AGENT_NAME = "summarization_agent"


def _group_by_sub_question(fact_check_results: List[Dict]) -> Dict[str, List[Dict]]:
    grouped = defaultdict(list)
    for fact in fact_check_results:
        grouped[fact.get("sub_question", "unknown")].append(fact)
    return dict(grouped)


def _build_summary_prompt(sub_question: str, facts: List[Dict]) -> str:
    confirmed = [f for f in facts if f["status"] == "CONFIRMED"]
    disputed = [f for f in facts if f["status"] == "DISPUTED"]
    unverified = [f for f in facts if f["status"] == "UNVERIFIED"]

    def fmt(fs):
        return "\n".join(f'- "{f["claim"]}" (reasoning: {f.get("reasoning", "")})' for f in fs) or "(none)"

    return f"""Write a short, honest research brief (3-6 sentences) answering: "{sub_question}"

Rules:
- State CONFIRMED facts plainly.
- For DISPUTED facts, explicitly say they are disputed and describe BOTH sides.
- For UNVERIFIED facts, hedge explicitly (e.g. "according to a single source...").
- Do not invent facts. Do not hide disputes.

CONFIRMED facts:
{fmt(confirmed)}

DISPUTED facts:
{fmt(disputed)}

UNVERIFIED facts:
{fmt(unverified)}

Write only the brief itself.
"""


def summarize_sub_topic(sub_question: str, facts: List[Dict]) -> Dict:
    if not facts:
        return {
            "sub_question": sub_question,
            "summary": "No verified information was found for this sub-topic despite an active search attempt.",
            "status": "no_facts",
            "fact_counts": {"CONFIRMED": 0, "DISPUTED": 0, "UNVERIFIED": 0},
        }

    counts = {"CONFIRMED": 0, "DISPUTED": 0, "UNVERIFIED": 0}
    for f in facts:
        counts[f["status"]] = counts.get(f["status"], 0) + 1

    try:
        summary_text = call_llm(_build_summary_prompt(sub_question, facts)).strip()
        status = "ok"
    except Exception as e:
        summary_text = f"Summary could not be generated due to an error ({type(e).__name__}). Raw fact counts: {counts}."
        status = "failed"
        log_agent_event(AGENT_NAME, "summary_failed", sub_question=sub_question, error=f"{type(e).__name__}: {e}")

    log_agent_event(AGENT_NAME, "summary_done", sub_question=sub_question, status=status, **counts)
    return {"sub_question": sub_question, "summary": summary_text, "status": status, "fact_counts": counts}


def summarization_node(state: Dict) -> Dict:
    """FIXED (Day 22, per Day 21's finding): iterates over the ORIGINAL plan,
    not just the sub-questions present in fact_check_results. Any planned
    sub-question with zero results now gets an explicit 'no information
    found' summary instead of silently vanishing from the final report."""
    fact_check_results = state.get("fact_check_results", [])
    plan = state.get("plan", [])

    if not fact_check_results and not plan:
        return {
            "errors": [{"agent": AGENT_NAME, "error": "no fact_check_results or plan to summarize"}],
            "model_usage_log": [{"agent": AGENT_NAME, "status": "skipped_no_input"}],
        }

    grouped = _group_by_sub_question(fact_check_results)

    # ensure every planned sub-question gets a summary, even if it has zero facts
    all_topics = list(dict.fromkeys(list(grouped.keys()) + plan))  # preserves order, dedupes

    summaries = [summarize_sub_topic(q, grouped.get(q, [])) for q in all_topics]

    return {
        "summaries": summaries,
        "model_usage_log": [{"agent": AGENT_NAME, "provider": "groq", "sub_topics": len(summaries)}],
    }

Overwriting agents/summarization_agent.py


In [1]:
from agents.summarization_agent import summarization_node

state = {
    "plan": ["topic A", "topic B", "topic C with zero facts"],
    "fact_check_results": [
        {"claim": "fact 1", "sub_question": "topic A", "status": "CONFIRMED", "reasoning": ""},
        {"claim": "fact 2", "sub_question": "topic B", "status": "CONFIRMED", "reasoning": ""},
        # note: "topic C with zero facts" has NO entries here, same as Day 21's real scenario
    ],
}

out = summarization_node(state)
print(f"Summaries produced: {len(out['summaries'])} (plan had {len(state['plan'])} topics)")
for s in out["summaries"]:
    print(f"[{s['status']}] {s['sub_question']}: {s['summary'][:80]}")

assert len(out["summaries"]) == 3, "FAIL: a planned topic still went missing"
assert any(s["status"] == "no_facts" for s in out["summaries"]), "FAIL: no explicit 'no facts' summary produced"
print("\nDay 21's bug is fixed: all 3 planned topics appear, the empty one is explicit, not silent")

2026-10-30 19:52:56,657 | INFO | research_system | [summarization_agent] summary_done sub_question=topic A status=ok CONFIRMED=1 DISPUTED=0 UNVERIFIED=0
2026-10-30 19:52:56,816 | INFO | research_system | [summarization_agent] summary_done sub_question=topic B status=ok CONFIRMED=1 DISPUTED=0 UNVERIFIED=0


Summaries produced: 3 (plan had 3 topics)
[ok] topic A: The only confirmed fact regarding "topic A" is "fact 1." No other information is
[ok] topic B: Regarding topic B, the only confirmed fact is "fact 2." No other information is 
[no_facts] topic C with zero facts: No verified information was found for this sub-topic despite an active search at

Day 21's bug is fixed: all 3 planned topics appear, the empty one is explicit, not silent


In [2]:
from agents.summarization_agent import summarization_node

state = {
    "plan": ["topic A", "topic B", "topic C with zero facts"],
    "fact_check_results": [
        {"claim": "fact 1", "sub_question": "topic A", "status": "CONFIRMED", "reasoning": ""},
        {"claim": "fact 2", "sub_question": "topic B", "status": "CONFIRMED", "reasoning": ""},
        # note: "topic C with zero facts" has NO entries here, same as Day 21's real scenario
    ],
}

out = summarization_node(state)
print(f"Summaries produced: {len(out['summaries'])} (plan had {len(state['plan'])} topics)")
for s in out["summaries"]:
    print(f"[{s['status']}] {s['sub_question']}: {s['summary'][:80]}")

assert len(out["summaries"]) == 3, "FAIL: a planned topic still went missing"
assert any(s["status"] == "no_facts" for s in out["summaries"]), "FAIL: no explicit 'no facts' summary produced"
print("\nDay 21's bug is fixed: all 3 planned topics appear, the empty one is explicit, not silent")

2026-10-30 20:17:19,468 | INFO | research_system | [summarization_agent] summary_done sub_question=topic A status=ok CONFIRMED=1 DISPUTED=0 UNVERIFIED=0
2026-10-30 20:17:19,658 | INFO | research_system | [summarization_agent] summary_done sub_question=topic B status=ok CONFIRMED=1 DISPUTED=0 UNVERIFIED=0


Summaries produced: 3 (plan had 3 topics)
[ok] topic A: The only confirmed fact regarding "topic A" is "fact 1." No other information is
[ok] topic B: Regarding topic B, the only confirmed fact is "fact 2." No other information is 
[no_facts] topic C with zero facts: No verified information was found for this sub-topic despite an active search at

Day 21's bug is fixed: all 3 planned topics appear, the empty one is explicit, not silent


In [3]:
%%writefile tests/test_summarization_agent.py
import agents.summarization_agent as sa


def test_summarize_empty_facts_no_llm_call(monkeypatch):
    called = {"value": False}
    def fake_call(p):
        called["value"] = True
        return "should not be called"
    monkeypatch.setattr(sa, "call_llm", fake_call)
    result = sa.summarize_sub_topic("q", [])
    assert result["status"] == "no_facts"
    assert called["value"] is False


def test_summarize_llm_failure_does_not_crash(monkeypatch):
    def boom(p):
        raise RuntimeError("down")
    monkeypatch.setattr(sa, "call_llm", boom)
    facts = [{"claim": "A", "status": "CONFIRMED", "reasoning": "", "sources": []}]
    result = sa.summarize_sub_topic("q", facts)
    assert result["status"] == "failed"


def test_summarize_counts_statuses_correctly(monkeypatch):
    monkeypatch.setattr(sa, "call_llm", lambda p: "a summary")
    facts = [
        {"claim": "A", "status": "CONFIRMED", "reasoning": "", "sources": []},
        {"claim": "B", "status": "CONFIRMED", "reasoning": "", "sources": []},
        {"claim": "C", "status": "DISPUTED", "reasoning": "", "sources": []},
        {"claim": "D", "status": "UNVERIFIED", "reasoning": "", "sources": []},
    ]
    result = sa.summarize_sub_topic("q", facts)
    assert result["fact_counts"] == {"CONFIRMED": 2, "DISPUTED": 1, "UNVERIFIED": 1}


def test_group_by_sub_question():
    facts = [
        {"claim": "A", "sub_question": "q1", "status": "CONFIRMED"},
        {"claim": "B", "sub_question": "q1", "status": "CONFIRMED"},
        {"claim": "C", "sub_question": "q2", "status": "DISPUTED"},
    ]
    grouped = sa._group_by_sub_question(facts)
    assert len(grouped) == 2


def test_node_aggregates_multiple_sub_topics(monkeypatch):
    monkeypatch.setattr(sa, "call_llm", lambda p: "a summary")
    state = {
        "plan": ["q1", "q2"],
        "fact_check_results": [
            {"claim": "A", "sub_question": "q1", "status": "CONFIRMED", "reasoning": "", "sources": []},
            {"claim": "B", "sub_question": "q2", "status": "CONFIRMED", "reasoning": "", "sources": []},
        ],
    }
    out = sa.summarization_node(state)
    assert len(out["summaries"]) == 2


def test_node_empty_fact_check_results_and_empty_plan():
    out = sa.summarization_node({"fact_check_results": [], "plan": []})
    assert out["errors"] and "summaries" not in out


def test_node_regression_day21_silent_topic_drop(monkeypatch):
    """Regression test for the Day 21 finding: a planned sub-question with
    zero fact_check_results must still appear in summaries, explicitly
    marked as having no information, not silently dropped."""
    monkeypatch.setattr(sa, "call_llm", lambda p: "a real summary")
    state = {
        "plan": ["topic A", "topic B", "topic with zero facts"],
        "fact_check_results": [
            {"claim": "fact 1", "sub_question": "topic A", "status": "CONFIRMED", "reasoning": "", "sources": []},
            {"claim": "fact 2", "sub_question": "topic B", "status": "CONFIRMED", "reasoning": "", "sources": []},
        ],
    }
    out = sa.summarization_node(state)
    assert len(out["summaries"]) == 3
    topics_covered = {s["sub_question"] for s in out["summaries"]}
    assert "topic with zero facts" in topics_covered
    empty_summary = next(s for s in out["summaries"] if s["sub_question"] == "topic with zero facts")
    assert empty_summary["status"] == "no_facts"

Overwriting tests/test_summarization_agent.py


In [4]:
%%writefile evaluate.py
"""Evaluation harness: runs a set of test queries through the full pipeline
and scores each report against concrete, automatically-checkable criteria —
not subjective quality, but structural guarantees the system should uphold."""
from typing import Dict, List


def eval_covers_all_planned_topics(result: Dict) -> Dict:
    """Every sub-question in the plan should have a corresponding summary.
    Directly tests the Day 21 finding."""
    planned = set(result.get("plan", []))
    covered = {s["sub_question"] for s in result.get("summaries", [])}
    missing = planned - covered
    return {
        "name": "covers_all_planned_topics",
        "passed": len(missing) == 0,
        "detail": f"{len(covered)}/{len(planned)} topics covered" + (f", missing: {missing}" if missing else ""),
    }


def eval_report_cites_sources(result: Dict) -> Dict:
    """The final report must contain the Confidence Appendix with real claim
    counts — not an empty/degenerate report."""
    report = result.get("final_report") or ""
    passed = "Confidence Appendix" in report and "Total claims reviewed" in report
    return {"name": "report_has_confidence_appendix", "passed": passed, "detail": f"report length: {len(report)} chars"}


def eval_disputes_are_surfaced(result: Dict) -> Dict:
    """If any fact was DISPUTED, the final report text must actually mention
    it — disputes shouldn't exist in fact_check_results but vanish from
    the narrative the user actually reads."""
    disputed = [f for f in result.get("fact_check_results", []) if f.get("status") == "DISPUTED"]
    if not disputed:
        return {"name": "disputes_surfaced_if_present", "passed": True, "detail": "no disputed claims this run (n/a)"}
    report = (result.get("final_report") or "").lower()
    mentions_dispute = any(w in report for w in ["dispute", "disagree", "conflict", "contradic"])
    return {
        "name": "disputes_surfaced_if_present",
        "passed": mentions_dispute,
        "detail": f"{len(disputed)} disputed claim(s) in state; dispute language in report: {mentions_dispute}",
    }


def eval_no_unhandled_errors(result: Dict) -> Dict:
    """Errors should be absorbed by fail-safes, not crash the pipeline —
    but this checks that if errors DID occur, the pipeline still completed."""
    has_errors = len(result.get("errors", [])) > 0
    completed = result.get("final_report") is not None
    return {
        "name": "completes_despite_errors",
        "passed": completed,
        "detail": f"errors: {len(result.get('errors', []))}, completed: {completed}",
    }


def eval_terminates_within_iteration_budget(result: Dict, max_expected: int = 7) -> Dict:
    """Sanity check against runaway loops — should finish well within the
    MAX_ITERATIONS safety cap, not right up against it."""
    iterations = result.get("iteration_count", 0)
    return {
        "name": "terminates_within_budget",
        "passed": iterations <= max_expected,
        "detail": f"used {iterations} iterations (budget: {max_expected})",
    }


ALL_CHECKS = [
    eval_covers_all_planned_topics,
    eval_report_cites_sources,
    eval_disputes_are_surfaced,
    eval_no_unhandled_errors,
    eval_terminates_within_iteration_budget,
]


def evaluate_run(result: Dict) -> List[Dict]:
    return [check(result) for check in ALL_CHECKS]


def print_eval_report(query: str, checks: List[Dict]):
    passed = sum(1 for c in checks if c["passed"])
    print(f"\n{'='*70}\nQUERY: {query}\nScore: {passed}/{len(checks)}\n{'='*70}")
    for c in checks:
        mark = "✅" if c["passed"] else "❌"
        print(f"{mark} {c['name']}: {c['detail']}")

Writing evaluate.py


In [ ]:
from graph.build_graph import build_graph
from evaluate import evaluate_run, print_eval_report

app = build_graph()

eval_query = "Compare the battery recycling strategies of Tesla, Redwood Materials, and CATL"
initial_state = {
    "query": eval_query, "plan": [], "next_agent": "", "iteration_count": 0,
    "needs_more_research": False, "search_results": [], "extracted_facts": [],
    "fact_check_results": [], "summaries": [], "errors": [], "final_report": None,
    "model_usage_log": [],
}
result = app.invoke(initial_state)

checks = evaluate_run(result)
print_eval_report(eval_query, checks)